# Module 02 Homework - End-to-End Workflow

## Overview
- Purpose: implement the official Module 02 vector search homework.
- Objectives: understand embeddings, cosine similarity, chunking, vector search, keyword search, hybrid search, and RRF.
- What: a reproducible notebook and Python workflow that compute the homework answers directly from the official lesson content.
- Why: the workflow is beginner-friendly, modular, and production-oriented.
- How: the notebook runs the full pipeline from environment setup to report generation.
- Expected Output: a generated JSON summary, markdown report, and optional figure.
- Intermediate Results: embedding values, cosine similarity, chunk counts, search rankings.
- Final Results: the computed answers to Questions 1–6.
- Interpretation: each method highlights a different retrieval behavior.
- Troubleshooting: rerun the first cell if the model or helper files are missing.
- Summary: the project is fully automated and generates the homework report on execution.

In [ ]:
import json
import sys
from pathlib import Path

root = Path.cwd().resolve()
if root.name == 'Homework':
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from Homework.homework import run_homework

summary = run_homework()
print(json.dumps(summary, indent=2))

## Section 1 - Questions 1 to 3

### Overview
- Objective: inspect the embedding, similarity, and chunking behavior.
- What: embed the query, compute cosine similarity, and identify the top chunk.
- Why: these steps form the foundation of vector search.
- How: use the ONNX embedder and overlap-based chunking.
- Expected Output: the answers to Questions 1–3.
- Intermediate Results: embedding preview, similarity score, chunk scores.
- Final Results: the exact values computed by the workflow.
- Interpretation: chunking improves retrieval granularity.
- Troubleshooting: ensure the ONNX model exists in the models folder.
- Summary: Questions 1–3 are computed directly from data and embeddings.

In [ ]:
from Homework.config import QUERY_Q1
from Homework.helper import build_embedder, load_documents

embedder = build_embedder('Homework/models/Xenova/all-MiniLM-L6-v2')
query_vector = embedder.encode(QUERY_Q1)
print('Q1 first value:', float(query_vector[0]))

documents = load_documents()
target_filename = '02-vector-search/lessons/07-sqlitesearch-vector.md'
target_document = next(doc for doc in documents if doc['filename'] == target_filename)
target_vector = embedder.encode(target_document['content'])
print('Q2 cosine similarity:', float(query_vector @ target_vector))

## Section 2 - Question 4

### Overview
- Objective: run MinSearch vector search.
- What: embed the evaluation query and search the chunked lessons with MinSearch.
- Why: vector search is the practical retrieval method used in production systems.
- How: build a VectorSearch index and query it with the embedding.
- Expected Output: the top retrieved filename for Question 4.
- Intermediate Results: search ranking list.
- Final Results: the answer to Question 4.
- Interpretation: MinSearch returns semantically relevant content.
- Troubleshooting: verify that the chunk vectors were built correctly.
- Summary: Question 4 is answered by the vector search ranking.

In [ ]:
from Homework.config import QUERY_Q4
from Homework.helper import create_chunks, embed_texts, build_vector_index, search_vector

chunks = create_chunks(documents)
chunk_vectors = embed_texts(embedder, [chunk['content'] for chunk in chunks])
vector_index = build_vector_index(chunk_vectors, chunks)
q4_vector = embedder.encode(QUERY_Q4)
q4_results = search_vector(vector_index, q4_vector, top_k=5)
print('Q4 results:', [doc['filename'] for doc in q4_results])

## Section 3 - Question 5

### Overview
- Objective: compare vector search and keyword search.
- What: run both retrieval methods for the PostgreSQL query and inspect the overlap.
- Why: vector search and keyword search capture different signals.
- How: build a text index and compare the top results to the vector search results.
- Expected Output: the file unique to vector search.
- Intermediate Results: both ranked lists.
- Final Results: the Question 5 answer.
- Interpretation: vector search can retrieve relevant content even when keywords differ.
- Troubleshooting: confirm that the chunk content is indexed properly.
- Summary: Question 5 highlights the difference between semantic and lexical retrieval.

In [ ]:
from Homework.config import QUERY_Q5
from Homework.helper import build_text_index, search_text

text_index = build_text_index(chunks)
q5_vector_results = search_vector(vector_index, embedder.encode(QUERY_Q5), top_k=5)
q5_keyword_results = search_text(text_index, QUERY_Q5, top_k=5)
q5_unique = next((doc['filename'] for doc in q5_vector_results if doc['filename'] not in {item['filename'] for item in q5_keyword_results}), None)
print('Q5 vector results:', [doc['filename'] for doc in q5_vector_results])
print('Q5 keyword results:', [doc['filename'] for doc in q5_keyword_results])
print('Q5 unique to vector:', q5_unique)

## Section 4 - Question 6

### Overview
- Objective: run hybrid search with RRF.
- What: fuse vector and keyword rankings.
- Why: hybrid search often produces stronger results than either strategy alone.
- How: apply Reciprocal Rank Fusion to the two ranked lists.
- Expected Output: the best document after fusion.
- Intermediate Results: the vector ranking, keyword ranking, and fused ranking.
- Final Results: the answer to Question 6.
- Interpretation: the fused ranking can elevate documents that are strong in both approaches.
- Troubleshooting: review the ranking lists if the result seems unexpected.
- Summary: Question 6 demonstrates how hybrid search improves retrieval.

In [ ]:
from Homework.config import QUERY_Q6
from Homework.helper import compute_rrf

q6_vector_results = search_vector(vector_index, embedder.encode(QUERY_Q6), top_k=5)
q6_keyword_results = search_text(text_index, QUERY_Q6, top_k=5)
q6_rrf_results = compute_rrf([q6_vector_results, q6_keyword_results], num_results=5)
print('Q6 vector:', [doc['filename'] for doc in q6_vector_results])
print('Q6 keyword:', [doc['filename'] for doc in q6_keyword_results])
print('Q6 RRF:', [doc['filename'] for doc in q6_rrf_results])

## Final Summary

### Overview
- Objective: present the final computed answers and generated artifacts.
- What: the notebook summarizes the answers and writes the report files.
- Why: the outputs should be reproducible and easy to inspect.
- How: the first cell already wrote the JSON and markdown report.
- Expected Output: comprehensive results in the outputs directory.
- Intermediate Results: the ranking tables and comparison outputs.
- Final Results: the final computed answers to the homework questions.
- Interpretation: the workflow produces the answers automatically from the lesson content.
- Troubleshooting: if artifacts are missing, rerun the first cell.
- Summary: the homework report is updated whenever the workflow is executed.

In [ ]:
print(json.dumps(summary['questions'], indent=2))
print('\nReport written to Homework/README.md')